# 🤖 Notebook 10: LLM Explanation Layer (Cầu Nối AI Kỹ Thuật & Nghiệp Vụ Ngân Hàng)

> **Mục tiêu**: Biến các chỉ số toán học trừu tượng (SHAP Values, Feature Importances, Risk Probabilities) thành **báo cáo giải trình tiếng Việt tự nhiên, chuẩn nghiệp vụ Ngân hàng** cho các chuyên viên phòng Chống Gian lận (Fraud/Compliance Analysts).
> 
> **Kiến trúc Pipeline**:
> `Raw Transaction JSON` ➡️ `Preprocessor` ➡️ `XGBoost Model` ➡️ `SHAP Engine` ➡️ `LLM Explainer Layer` ➡️ `Báo Cáo Nghiệp Vụ`


In [1]:
import sys
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
import xgboost as xgb

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed_data, load_raw_data, temporal_split
from src.explainability import calculate_shap_values, get_top_influential_features
from src.llm_explainer import generate_fraud_explanation, rule_based_explainer_vi

print("Loading Data and Models...")
df_test = load_processed_data("test.parquet")
X_test = df_test.drop(columns=['fraud_bool', 'month'], errors='ignore')
y_test = df_test['fraud_bool'].values

estimator = xgb.XGBClassifier()
estimator.load_model(str(PROJECT_ROOT / "models" / "xgboost_best.json"))

# Lấy 1,000 mẫu để tính SHAP
base_value, shap_values, X_sample = calculate_shap_values(estimator, X_test, max_samples=1000)
y_sample = y_test[X_sample.index]

OPTIMAL_THRESHOLD = 0.48
probs_sample = estimator.predict_proba(X_sample)[:, 1]
preds_sample = (probs_sample >= OPTIMAL_THRESHOLD).astype(int)

print(f"Sẵn sàng thử nghiệm trên {len(X_sample)} giao dịch!")


Loading Data and Models...
Sẵn sàng thử nghiệm trên 1000 giao dịch!


## 1. Tìm Kiếm Các Hồ Sơ Đại Diện (Case Selection)
Chúng ta sẽ trích xuất 3 tình huống nghiệp vụ thực tế:
1. **Case 1 (True Positive)**: Gian lận thực sự bị phát hiện.
2. **Case 2 (False Positive)**: Khách hàng chân chính bị hệ thống nghi ngờ nhầm.
3. **Case 3 (True Negative)**: Khách hàng an toàn, phê duyệt thẳng.


In [2]:
tp_idx = np.where((preds_sample == 1) & (y_sample == 1))[0][0]
fp_idx = np.where((preds_sample == 1) & (y_sample == 0))[0][0]
tn_idx = np.where((preds_sample == 0) & (y_sample == 0))[0][0]

print(f"Case 1 (True Positive)  : Index {tp_idx:3d} | Risk Score = {probs_sample[tp_idx]:.2%}")
print(f"Case 2 (False Positive) : Index {fp_idx:3d} | Risk Score = {probs_sample[fp_idx]:.2%}")
print(f"Case 3 (True Negative)  : Index {tn_idx:3d} | Risk Score = {probs_sample[tn_idx]:.2%}")


Case 1 (True Positive)  : Index  28 | Risk Score = 98.11%
Case 2 (False Positive) : Index   0 | Risk Score = 70.30%
Case 3 (True Negative)  : Index   1 | Risk Score = 9.03%


## 2. Thử Nghiệm Case 1: Phát Hiện Gian Lận Thực Sự (True Positive)


In [3]:
# 1. Trích xuất Top 5 lý do kỹ thuật từ SHAP
top_reasons_tp = get_top_influential_features(
    shap_values_row=shap_values[tp_idx],
    feature_names=list(X_sample.columns),
    feature_values_row=X_sample.iloc[tp_idx].values,
    top_k=5
)

# 2. Sinh báo cáo nghiệp vụ tiếng Việt
explanation_tp = generate_fraud_explanation(
    risk_score=float(probs_sample[tp_idx]),
    threshold=OPTIMAL_THRESHOLD,
    top_reasons=top_reasons_tp,
    use_llm=True # Sẽ dùng Gemini nếu có API key, tự động dùng Domain NLG nếu không có key
)

print(explanation_tp)


[LLM Warning] Lỗi gọi OpenAI API: HTTP Error 429: Too Many Requests. Đang chuyển fallback...
🚨 **CẢNH BÁO GIAN LẬN (Điểm rủi ro: 98.1% - Vượt ngưỡng 48.0%)**

Hệ thống AI đề xuất **CHẶN / CHUYỂN REVIEW THỦ CÔNG** do phát hiện các dấu hiệu bất thường sau:
1. **Hạn mức tín dụng đề xuất cao bất thường so với hồ sơ** *(Đóng góp SHAP: +0.867)*
2. **Tình trạng cư trú thuộc nhóm rủi ro cao (nhà thuê ngắn hạn/không cố định)** *(Đóng góp SHAP: +0.718)*
3. **Đặc trưng `device_os` (giá trị: 3.0) đóng góp tăng nguy cơ rủi ro** *(Đóng góp SHAP: +0.530)*
4. **Điểm rủi ro tín dụng nội bộ ở mức cảnh báo cao** *(Đóng góp SHAP: +0.438)*
5. **Ngày sinh của hồ sơ này được sử dụng với nhiều email khác nhau trong 4 tuần qua (dấu hiệu giả mạo danh tính)** *(Đóng góp SHAP: +0.360)*

📋 **Khuyến nghị cho Compliance Officer**: Yêu cầu khách hàng xác thực sinh trắc học bổ sung (eKYC) hoặc liên hệ đối chiếu nguồn thu nhập trước khi mở tài khoản.


## 3. Thử Nghiệm Case 2: Phân Tích Bắt Oan (False Positive)
> Trường hợp này rất quan trọng trong dịch vụ khách hàng: Khi khách hàng gọi lên khiếu nại "Tại sao tôi bị từ chối mở thẻ?", nhân viên tổng đài cần biết chính xác lý do để hỗ trợ gỡ phong tỏa.


In [4]:
top_reasons_fp = get_top_influential_features(
    shap_values_row=shap_values[fp_idx],
    feature_names=list(X_sample.columns),
    feature_values_row=X_sample.iloc[fp_idx].values,
    top_k=5
)

explanation_fp = generate_fraud_explanation(
    risk_score=float(probs_sample[fp_idx]),
    threshold=OPTIMAL_THRESHOLD,
    top_reasons=top_reasons_fp,
    use_llm=True
)

print(explanation_fp)


[LLM Warning] Lỗi gọi OpenAI API: HTTP Error 429: Too Many Requests. Đang chuyển fallback...
🚨 **CẢNH BÁO GIAN LẬN (Điểm rủi ro: 70.3% - Vượt ngưỡng 48.0%)**

Hệ thống AI đề xuất **CHẶN / CHUYỂN REVIEW THỦ CÔNG** do phát hiện các dấu hiệu bất thường sau:
1. **Thiết bị này đã từng liên kết với nhiều tài khoản email khác nhau trong 8 tuần qua (dấu hiệu nông trại tài khoản / botnet)** *(Đóng góp SHAP: +0.780)*
2. **Đặc trưng `device_os` (giá trị: 0.0) đóng góp giảm nguy cơ rủi ro** *(Đóng góp SHAP: -0.626)*
3. **Thời gian cư trú tại địa chỉ hiện tại ổn định, lâu năm** *(Đóng góp SHAP: -0.470)*
4. **Tên khách hàng và địa chỉ email có độ tương đồng rất thấp (nghi ngờ email tạo tự động)** *(Đóng góp SHAP: +0.447)*
5. **Tình trạng sở hữu nhà ở ổn định** *(Đóng góp SHAP: -0.410)*

📋 **Khuyến nghị cho Compliance Officer**: Yêu cầu khách hàng xác thực sinh trắc học bổ sung (eKYC) hoặc liên hệ đối chiếu nguồn thu nhập trước khi mở tài khoản.


## 4. Thử Nghiệm Case 3: Hồ Sơ An Toàn (True Negative)


In [5]:
top_reasons_tn = get_top_influential_features(
    shap_values_row=shap_values[tn_idx],
    feature_names=list(X_sample.columns),
    feature_values_row=X_sample.iloc[tn_idx].values,
    top_k=5
)

explanation_tn = generate_fraud_explanation(
    risk_score=float(probs_sample[tn_idx]),
    threshold=OPTIMAL_THRESHOLD,
    top_reasons=top_reasons_tn,
    use_llm=True
)

print(explanation_tn)


[LLM Warning] Lỗi gọi OpenAI API: HTTP Error 429: Too Many Requests. Đang chuyển fallback...
✅ **GIAO DỊCH HỢP LỆ (Điểm rủi ro: 9.0% - Dưới ngưỡng 48.0%)**

Hệ thống AI đề xuất **TỰ ĐỘNG PHÊ DUYỆT**. Các yếu tố đánh giá an toàn:
1. **Đặc trưng `prev_address_months_count_is_missing` (giá trị: 0.0) đóng góp giảm nguy cơ rủi ro** *(Đóng góp SHAP: -0.674)*
2. **Thời gian cư trú tại địa chỉ hiện tại ổn định, lâu năm** *(Đóng góp SHAP: -0.589)*
3. **Đặc trưng `phone_home_valid` (giá trị: 1.0) đóng góp giảm nguy cơ rủi ro** *(Đóng góp SHAP: -0.553)*
4. **Tình trạng sở hữu nhà ở ổn định** *(Đóng góp SHAP: -0.507)*
5. **Đặc trưng `device_os` (giá trị: 2.0) đóng góp giảm nguy cơ rủi ro** *(Đóng góp SHAP: -0.401)*

📋 **Khuyến nghị cho Compliance Officer**: Hồ sơ đủ điều kiện giải ngân/cấp hạn mức tự động.
